<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

# ML-2 : Préparation des données et ingénierie des features

**Navigation** : [Index](README.md) | [<< ML-1](ML-1-Introduction.ipynb) | [Suivant >>](ML-3-Entrainement%26AutoML.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Charger des données depuis un fichier CSV avec TextLoader
2. Explorer et manipuler un IDataView
3. Appliquer des transformations : encoding, normalisation, concaténation
4. Construire un pipeline de feature engineering complet

### Prérequis
- ML-1-Introduction complété
- Fichier taxi-fare.csv disponible

### Durée estimée : 40-50 minutes

---

# Préparation des Données et Ingénierie des Caractéristiques

Les données sont cruciales pour entraîner et préparer un modèle. Dans ce notebook, nous allons couvrir comment charger des données dans ML.NET et s'assurer qu'elles sont dans le bon format pour que ML.NET puisse les utiliser.

## Chargement des données dans ML.NET

### Qu'est-ce qu'un IDataView?

Le [IDataView](https://docs.microsoft.com/dotnet/api/microsoft.ml.idataview?view=ml-dotnet) est le format de données que ML.NET charge pour l'entraînement. Il s'agit d'un ensemble d'interfaces et de composants qui fournissent un traitement efficace et compositionnel des données schématisées pour les applications de machine learning et d'analytique avancée. Il est conçu pour gérer de manière efficace les données de haute dimension et les grands ensembles de données.

### Comment créer un IDataView

Vous pouvez créer un IDataView en utilisant l'une des méthodes de chargement des données suivantes :

- TextLoader
- LoadFromEnumerable
- DatabaseLoader
- LoadFromTextFile

Pour plus de documentation et d'exemples, consultez [Charger les données à partir de fichiers et d'autres sources](https://docs.microsoft.com/dotnet/machine-learning/how-to-guides/load-data-ml-net).

In [1]:
#r "nuget: Microsoft.ML, 5.0.0"

using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Transforms;

MLContext mlContext = new MLContext();

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.ML, 5.0.0

### Télécharger ou localiser les données

Le code suivant essaie de localiser le fichier de données dans quelques emplacements connus ou le téléchargera à partir de l'emplacement GitHub connu.

In [2]:
using System;
using System.IO;
using System.Net;

string EnsureDataSetDownloaded(string fileName)
{
    // Ce chemin est utilisé si le dépôt a été cloné.
    var filePath = Path.Combine(Directory.GetCurrentDirectory(),"data", fileName);

    if (!File.Exists(filePath))
    {
        // Ce chemin est utilisé si le fichier a déjà été téléchargé.
        filePath = Path.Combine(Directory.GetCurrentDirectory(), fileName);
    }

    if (!File.Exists(filePath))
    {
        using (var client = new WebClient())
        {
            client.DownloadFile($"https://raw.githubusercontent.com/dotnet/csharp-notebooks/main/machine-learning/data/{fileName}", filePath);
        }
        Console.WriteLine($"Téléchargé {fileName} à : {filePath}");
    }
    else
    {
        Console.WriteLine($"{fileName} trouvé ici : {filePath}");
    }

    return filePath;
}


(18,29): warning SYSLIB0014: 'WebClient.WebClient()' est obsolète : 'WebRequest, HttpWebRequest, ServicePoint, and WebClient are obsolete. Use HttpClient instead.' (https://aka.ms/dotnet-warnings/SYSLIB0014)



### Chargement depuis un fichier

Un [TextLoader](https://docs.microsoft.com/dotnet/api/microsoft.ml.data.textloader?view=ml-dotnet) peut charger un fichier structuré dans un IDataView. Les informations structurées sont représentées comme des colonnes et des lignes de données.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [3]:
public class ModelInput
{
    [LoadColumn(0)]
    [ColumnName(@"vendor_id")]
    public string Vendor_id { get; set; }

    [LoadColumn(1)]
    [ColumnName(@"rate_code")]
    public float Rate_code { get; set; }

    [LoadColumn(2)]
    [ColumnName(@"passenger_count")]
    public float Passenger_count { get; set; }

    [LoadColumn(3)]
    [ColumnName(@"trip_time_in_secs")]
    public float Trip_time_in_secs { get; set; }

    [LoadColumn(4)]
    [ColumnName(@"trip_distance")]
    public float Trip_distance { get; set; }

    [LoadColumn(5)]
    [ColumnName(@"payment_type")]
    public string Payment_type { get; set; }

    [LoadColumn(6)]
    [ColumnName(@"fare_amount")]
    public float Fare_amount { get; set; }
}

MLContext mlContext = new MLContext();

var trainDataPath = EnsureDataSetDownloaded("taxi-fare.csv");

// Créer un TextLoader basé sur le type ModelInput.
TextLoader textLoader = mlContext.Data.CreateTextLoader<ModelInput>(separatorChar: ',', hasHeader: true);

// Charger les données dans un IDataView. La méthode Load() peut prendre en charge plusieurs fichiers.
// Les fichiers doivent avoir le même caractère séparateur, en-tête, noms de colonnes, etc.
IDataView data = textLoader.Load(trainDataPath);

data.Preview(1)

Error: System.IO.FileNotFoundException: Could not load file or assembly 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'. Le fichier spécifié est introuvable.
File name: 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'
   at Submission#4.<<Initialize>>d__0.MoveNext()
   at System.Runtime.CompilerServices.AsyncMethodBuilderCore.Start[TStateMachine](TStateMachine& stateMachine)
   at Submission#4.<Initialize>()
   at Submission#4.<Factory>(Object[] submissionArray)
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

## Chargement d'une collection en mémoire

ML.NET prend en charge le chargement des données à partir d'une collection en mémoire. Cela facilite le chargement à partir d'un fichier JSON ou XML en utilisant C#. Apprenez comment [désérialiser JSON avec C#](https://docs.microsoft.com/dotnet/standard/serialization/system-text-json-how-to?pivots=dotnet-6-0#how-to-read-json-as-net-objects-deserialize) ou utilisez [XML serializer](https://docs.microsoft.com/dotnet/api/system.xml.serialization?view=net-6.0) pour obtenir ces fichiers en mémoire.

Une fois que vous avez la collection de données en mémoire, vous pouvez la charger dans ML.NET avec la méthode [LoadFromEnumerable](https://docs.microsoft.com/dotnet/api/microsoft.ml.dataoperationscatalog.loadfromenumerable?view=ml-dotnet).


In [4]:
ModelInput[] inMemoryCollection = new ModelInput[]
{
    new ModelInput
    {
        Vendor_id = "CMT",
        Rate_code = 1,
        Passenger_count = 1,
        Trip_time_in_secs = 1271,
        Trip_distance = 3.8f,
        Payment_type = "CRD",
        Fare_amount = 17.5f,
    },
    new ModelInput
    {
        Vendor_id = "VST",
        // Rate_code manquant
        Passenger_count = 1,
        Trip_time_in_secs = 474,
        Trip_distance = 1.5f,
        Payment_type = "CSH",
        Fare_amount = 8,
    }
};

// Créer un MLContext
MLContext mlContext = new MLContext();

// Charger les données
IDataView data = mlContext.Data.LoadFromEnumerable<ModelInput>(inMemoryCollection);

// Afficher les données chargées
var preview = data.Preview();
display(preview);

[Cellule non executee - erreur environnement .NET]


### Différence entre DataFrame et IDataView

Vous avez peut-être entendu parler du type [DataFrame](https://docs.microsoft.com/dotnet/api/microsoft.data.analysis.dataframe?view=ml-dotnet-preview). C'est un autre outil pour charger, visualiser et manipuler des données, courant dans les notebooks. Il implémente un IDataView, il peut donc être facilement passé à ML.NET.

DataFrame et IDataView sont très similaires en ce sens qu'ils sont tous deux des moyens de représenter des données sous forme tabulaire et d'appliquer des transformations. Voici quelques différences clés :

- DataFrame ne prend en charge que le chargement de fichiers délimités.
- DataFrame fonctionne en mémoire, vous êtes donc limité par la quantité de mémoire de votre PC.

Le DataFrame est recommandé pour effectuer des tâches telles que l'analyse exploratoire des données sur un échantillon de vos données. Consultez le notebook de référence [REF-Data Processing](https://github.com/dotnet/csharp-notebooks/blob/main/machine-learning/REF-Data%20Processing.ipynb) pour un exemple d'utilisation des DataFrames pour manipuler un fichier de données pour l'entraînement.

IDataView est recommandé pour l'entraînement sur des ensembles de données plus grands, et ce qui est utilisé pour les exemples dans ce notebook. Les ensembles de données plus grands dans ce cas sont définis comme des ensembles de données qui ne peuvent pas tenir en mémoire.

## Transformations des données

ML.NET prend en charge une variété de transformations de données qui convertiront les données dans le format requis et vous aideront à apporter des corrections à vos données. Parmi les opérations courantes, citons la manipulation des colonnes, la normalisation des valeurs, le remplacement des valeurs manquantes, la conversion des valeurs, etc.

Pour plus d'informations, consultez [Transformations des données](https://docs.microsoft.com/dotnet/machine-learning/resources/transforms).

### Données catégorielles

L'encodage one-hot est une transformation importante pour les données contenant des catégories. Les algorithmes ML nécessitent des données numériques, ils ne savent pas comment traiter les chaînes représentant des catégories. Les colonnes vendor_id et payment_type sont catégoriques, le fournisseur peut être "CMD" ou "VST" et le paiement peut être "CReDit" ou "CaSH". L'encodage one-hot prend les valeurs de chaîne passées et les convertit en données numériques.

In [5]:
var pipeline = mlContext.Transforms.Categorical.OneHotEncoding(
    new[] 
    { new InputOutputColumnPair(@"vendor_id"), 
    new InputOutputColumnPair(@"payment_type")},
    OneHotEncodingEstimator.OutputKind.Binary);
Console.WriteLine("Pipeline OneHotEncoding configure.");


[Cellule non executee - erreur environnement .NET]


Testons la transformation ci-dessus sur les colonnes vendor_id et payment_type. Le résultat devrait être une valeur vectorielle pour chaque catégorie. Dans le cas de Vendor_Id, CMT devient `000` et VST devient `001`. Nous allons créer une nouvelle classe ModelInputTransformed pour les nouveaux types convertis.

In [6]:
using System.Numerics;

public class ModelInputTransformed
{
    [LoadColumn(0)]
    [ColumnName(@"vendor_id")]
    public VBuffer<Single> Vendor_id { get; set; }

    [LoadColumn(1)]
    [ColumnName(@"rate_code")]
    public float Rate_code { get; set

; }

    [LoadColumn(2)]
    [ColumnName(@"passenger_count")]
    public float Passenger_count { get; set; }

    [LoadColumn(3)]
    [ColumnName(@"trip_time_in_secs")]
    public float Trip_time_in_secs { get; set; }

    [LoadColumn(4)]
    [ColumnName(@"trip_distance")]
    public float Trip_distance { get; set; }

    [LoadColumn(5)]
    [ColumnName(@"payment_type")]
    public VBuffer<Single> Payment_type { get; set; }

    [LoadColumn(6)]
    [ColumnName(@"fare_amount")]
    public float Fare_amount { get; set; }
}

// Exécuter la transformation
IDataView transformedData = pipeline.Fit(data).Transform(data);
var convertedData = mlContext.Data.CreateEnumerable<ModelInputTransformed>(transformedData, true);

// Encodage one-hot de deux colonnes 'vendor_id' et 'payment_type'.
Console.WriteLine("Vendor_Id" +"\t" + "Payment_Type"); 
foreach (ModelInputTransformed item in convertedData.Take(10))
{    
    Console.WriteLine("{0}\t\t{1}", string.Join(" ", item.Vendor_id.DenseValues()),
                    string.Join(" ", item.Payment_type.DenseValues()));
}

[Cellule non executee - erreur environnement .NET]


### Remplacer les valeurs manquantes

Une autre opération courante est de remplacer les valeurs manquantes. Ici, nous utilisons le mode de remplacement par défaut, qui remplace la valeur par la valeur par défaut de son type.


In [7]:
pipeline.Append(mlContext.Transforms.ReplaceMissingValues(
    new[] { new InputOutputColumnPair(@"rate_code", @"rate_code"), 
    new InputOutputColumnPair(@"passenger_count", @"passenger_count"), 
    new InputOutputColumnPair(@"trip_time_in_secs", @"trip_time_in_secs"), 
    new InputOutputColumnPair(@"trip_distance", @"trip_distance") }));
Console.WriteLine("Transforms ReplaceMissingValues ajoutes au pipeline.");


[Cellule non executee - erreur environnement .NET]


Encore une fois, exécutons la transformation et jetons un coup d'œil à la valeur remplie. Il nous manquait le rate_code pour le deuxième objet factice.

In [8]:
IDataView  transformedData = pipeline.Fit(data).Transform(data);
var convertedData = mlContext.Data.CreateEnumerable<ModelInputTransformed>(transformedData, true);

"Rate_code: " + convertedData.ElementAt(1).Rate_code

[Cellule non executee - erreur environnement .NET]


### Concaténer les colonnes de caractéristiques

Concaténons maintenant toutes nos colonnes de caractéristiques en une seule colonne vectorielle. De nombreux formateurs ML s'attendent à ce que les caractéristiques soient de type vecteur, car l'application d'opérations sur un vecteur est plus efficace.

In [9]:
pipeline.Append(mlContext.Transforms.Concatenate(
    @"Features", new[] { @"vendor_id", @"payment_type", @"rate_code", @"passenger_count", @"trip_time_in_secs", @"trip_distance" }));
Console.WriteLine("Concatenation des features configuree.");


[Cellule non executee - erreur environnement .NET]


Nous avons maintenant un IDataView chargé et un pipeline prêt pour l'entraînement.

## Exercices supplementaires

Les exercices suivants approfondissent les transformations de donnees et le feature engineering avec ML.NET.

### Exercice 1 : Featurisation de texte personnalisee

Creez un pipeline utilisant `FeaturizeText` avec des options personnalisees (n-grams de mots, n-grams de caracteres) et comparez les performances avec les parametres par defaut.

**Objectifs** :
1. Preparer un jeu de donnees contenant du texte (avis clients, commentaires, etc.)
2. Construire un pipeline avec `FeaturizeText` par defaut et entrainer un modele
3. Construire un second pipeline avec `FeaturizeTextOptions` personnalise (n-grams, char n-grams)
4. Comparer l'accuracy des deux modeles et analyser l'impact de la featurisation

**Indice** : Utilisez `new FeaturizeTextOptions { WordFeatureExtractor = new WordBagEstimator.Options { NgramLength = 2 }, CharFeatureExtractor = new WordBagEstimator.Options { NgramLength = 3 } }`. Observez comment les n-grams capturent des sequences de mots que les unigrams ignorent.

In [10]:
// Exercice 1 : Featurisation de texte personnalisee
// TODO etudiant : Creez un pipeline avec FeaturizeText et comparez avec les parametres par defaut
// Indice : utilisez FeaturizeTextOptions pour configurer word n-grams, char n-grams
// Etape 1 : chargez un jeu de donnees avec du texte (par ex. des avis produits)
// Etape 2 : creez un pipeline avec FeaturizeText par defaut et entrainez un modele
// Etape 3 : creez un second pipeline avec FeaturizeTextOptions personnalises
// Etape 4 : comparez l'accuracy des deux modeles

Console.WriteLine("Exercice a completer : featurisation de texte personnalisee");

Exercice a completer


### Exercice 2 : Feature cross - combinaison de features categorielles

Combinez deux variables categorielles en une seule feature croisee pour capturer leurs interactions, puis comparez les performances du modele avec et sans cette transformation.

**Objectifs** :
1. Ajouter une colonne categorielle supplementaire (ex : saison) aux donnees taxi
2. Concatener `vendor_id` et la nouvelle colonne en un seul feature croise via `mlContext.Transforms.Expression` ou `CustomMapping`
3. Entrainer un modele avec le cross-feature et un modele sans
4. Comparer les metriques de regression (R2, RMSE) entre les deux approches

**Indice** : Un cross-feature capture les interactions entre variables. Par exemple, vendor "CMT" en ete peut avoir un comportement different de vendor "VST" en hiver. Concatenez les deux colonnes string puis appliquez un OneHotEncoding sur le resultat.

In [11]:
// Exercice 2 : Feature cross - combinaison de features categorielles
// TODO etudiant : Combinez deux features categorielles en une seule feature croisee
// Indice : concatenez deux colonnes string avant le OneHotEncoding
// Etape 1 : ajoutez une colonne "season" (ete, hiver, printemps, automne) aux donnees taxis
// Etape 2 : combinez vendor_id et season en une seule colonne via une expression custom
// Etape 3 : entrainez un modele avec et sans le cross-feature
// Etape 4 : comparez les metriques (R2, RMSE) dans les deux cas

Console.WriteLine("Exercice a completer : feature cross");

Exercice a completer


## Exercice : Feature engineering pour la prédiction de prix immobiliers

En vous basant sur l'exemple des taxis, créez un pipeline de préparation de données pour prédire le **prix de vente** de maisons.

### Objectifs
1. Définir la classe `HouseInput` avec les colonnes : quartier (catégoriel), surface (numérique), nb_chambres (numérique), prix (label)
2. Appliquer l'encodage one-hot sur la colonne catégorielle `quartier`
3. Remplacer les valeurs manquantes pour les colonnes numériques
4. Concaténer toutes les features en une seule colonne `Features`

### Indices
- Utilisez `[LoadColumn]` pour mapper les colonnes CSV
- OneHotEncoding pour les catégories (quartier)
- ReplaceMissingValues pour les valeurs numériques manquantes
- Concatenate pour créer le vecteur de features final

In [12]:
// Exercice : Feature engineering pour maisons
// Complétez les parties TODO pour créer un pipeline de préparation

// TODO: Définir la classe HouseInput avec les attributs LoadColumn
public class HouseInput
{
    // Colonne 0: quartier (string, catégoriel)
    // Colonne 1: surface en m² (float)
    // Colonne 2: nb_chambres (float)
    // Colonne 3: prix (float, label)
}

// TODO: Créer un MLContext
MLContext houseContext = null;

// TODO: Définir les données d'exemple (3-4 maisons)
HouseInput[] houses = 
{
    // Exemple: { Quartier = "Centre", Surface = 120, NbChambres = 4, Prix = 350000 },
    // Ajoutez d'autres exemples avec différents quartiers
};

// TODO: Charger les données en IDataView
IDataView houseData = null;

// TODO: Créer le pipeline avec OneHotEncoding pour le quartier
IEstimator<ITransformer> housePipeline = null;

// TODO: Ajouter ReplaceMissingValues pour les colonnes numériques
// housePipeline = housePipeline.Append(...)

// TODO: Ajouter Concatenate pour créer la colonne Features
// housePipeline = housePipeline.Append(...)

// TODO: Appliquer les transformations et afficher le résultat
IDataView transformedHouses = null;

if (transformedHouses != null)
{
    var preview = transformedHouses.Preview();
    display(preview);
}
else
{
    Console.WriteLine("Exercice a completer : implementez les TODO ci-dessus");
}


Exercice a completer


## Résumé

Ce notebook a couvert la préparation des données et l'ingénierie des features avec ML.NET :

| Concept | Description |
|---------|-------------|
| TextLoader | Chargement structuré depuis un fichier CSV |
| LoadFromEnumerable | Chargement depuis une collection en mémoire |
| OneHotEncoding | Conversion des variables catégorielles en vecteurs binaires |
| ReplaceMissingValues | Remplacement des valeurs manquantes |
| Concatenate | Fusion de colonnes en un vecteur de features |

**Points clés** :
- IDataView est le format central pour toutes les données ML.NET
- Les transformations sont définies dans un pipeline et appliquées avec `Fit().Transform()`
- L'encodage one-hot est indispensable pour les colonnes catégorielles
- La concaténation des features prépare les données pour l'entraînement

**Navigation** : [Index](README.md) | [<< ML-1 Introduction](ML-1-Introduction.ipynb) | [Suivant >> ML-3 Entrainement&AutoML](ML-3-Entrainement%26AutoML.ipynb)